# C07_E01 — Lugar de raíces y diseño clásico

**Capítulo:** 7 — Lugar de raíces y diseño clásico  
**Identificador:** `C07_E01`  
**Objetivo pedagógico:** Diseñar Kc para un amortiguamiento objetivo mediante LDR.  
**Librerías:** python-control, numpy, matplotlib

## Ejemplo industrial

Diseño de ganancia proporcional para un lazo de nivel con planta de segundo orden.

---

> **Aviso de uso responsable.** Este notebook es didáctico. No es código de producción. Cualquier implementación real requiere validación independiente. Ver `docs/politica_uso_responsable.md`.

## 1. Sistema y trazado del LDR

In [1]:
import control as ct
import numpy as np
import matplotlib.pyplot as plt
import os

# Planta de segundo orden
G = ct.tf([0.5], [20.0, 12.0, 1.0])

fig = plt.figure(figsize=(7, 6))
ct.root_locus(G, plot=True, grid=True)
plt.title("C07_E01 — Lugar de raíces de G(s)")
out = '../../figures/cap07/fig_C07_F01_root_locus.png'
os.makedirs(os.path.dirname(out), exist_ok=True)
plt.savefig(out, dpi=120, bbox_inches='tight'); plt.show()

/sessions/laughing-tender-dirac/.local/lib/python3.10/site-packages/control/rlocus.py:202: FutureWarning: root_locus() return value of roots, gains is deprecated; use root_locus_map()
  warnings.warn(
Ignoring fixed x limits to fulfill fixed data aspect with adjustable data limits.


## 2. Búsqueda de Kc para un amortiguamiento objetivo

In [2]:
# Barrido manual de Kc para encontrar zeta objetivo
kvect = np.linspace(0.1, 20, 200)
roots, _ = ct.root_locus(G, kvect=kvect, plot=False)

# Para cada Kc, calcular zeta efectivo del polo dominante complejo
def zeta_dominante(rs):
    rs_complex = rs[np.imag(rs) != 0]
    if len(rs_complex) == 0: return None
    r = rs_complex[np.argmin(np.abs(np.imag(rs_complex)))]
    wn = abs(r); zeta = -np.real(r)/wn
    return zeta

z_target = 0.7
zetas = [zeta_dominante(r) for r in roots]
zetas_arr = np.array([z if z is not None else np.nan for z in zetas])
idx = np.nanargmin(np.abs(zetas_arr - z_target))
print(f"Kc para zeta≈{z_target}: Kc = {kvect[idx]:.3f}")

Kc para zeta≈0.7: Kc = 5.300


/sessions/laughing-tender-dirac/tmp/ipykernel_67/4011686961.py:3: FutureWarning: keyword 'kvect' is deprecated; use 'gains'
  roots, _ = ct.root_locus(G, kvect=kvect, plot=False)
/sessions/laughing-tender-dirac/.local/lib/python3.10/site-packages/control/rlocus.py:202: FutureWarning: root_locus() return value of roots, gains is deprecated; use root_locus_map()
  warnings.warn(


## 3. Interpretación

El LDR muestra cómo se mueven los polos del lazo cerrado al variar Kc. Para esta planta de segundo orden, la búsqueda numérica devuelve la Kc que sitúa los polos sobre la línea de ζ ≈ 0,7 (≈ 5% sobredisparo). El LDR es una herramienta de razonamiento; en procesos con retardo dominante pierde valor predictivo y conviene complementar con análisis frecuencial (Capítulo 6).